# Preprocess — agent-final-2 (Appliances dataset)

Run **all cells top to bottom**. No separate pipeline scripts needed.

Steps:
1. Load & audit distributions
2. Correlation → decide drops
3. Preprocess → save `10min` + `hourly` tracks
4. Validate window sizes

Target: **Appliances** (Wh). Raw file: `energydata_complete.csv`.

In [ ]:
# Optional: Colab drive (skip locally)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import json
import os
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import MinMaxScaler

TARGET = 'Appliances'
TEST_RATIO = 0.18
HIGH_R_THRESHOLD = 0.85
LEAKAGE_R_THRESHOLD = 0.95
MUST_DROP = ['rv1', 'rv2']
SUM_COLS = ['Appliances', 'lights']
WINDOWS = {
    '10min': [1, 6, 12, 36, 72, 144, 288],
    'hourly': [1, 4, 8, 12, 24, 48, 72, 168],
}

# --- paths (Colab or local) ---
ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Shared-Colab-Storage/agent-final-2'),
    Path('/content/drive/MyDrive/agent-final-2'),
    Path('..').resolve(),
    Path('.').resolve().parent,
]
ROOT = next((p for p in ROOT_CANDIDATES if (p / 'outputs').exists() or (p / 'pipeline').exists()), ROOT_CANDIDATES[-1])

RAW_CANDIDATES = [
    ROOT.parent / 'energydata_complete.csv',
    Path('/content/drive/MyDrive/Shared-Colab-Storage/energydata_complete.csv'),
    Path('/content/drive/MyDrive/energydata_complete.csv'),
]
RAW = next((p for p in RAW_CANDIDATES if p.exists()), RAW_CANDIDATES[0])

OUT = ROOT / 'outputs'
for sub in ['step01', 'step02', 'step03', 'step04', 'preprocess/10min', 'preprocess/hourly']:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('RAW:', RAW, 'exists:', RAW.exists())
print('OUT:', OUT)

In [ ]:
def load_raw():
    df = pd.read_csv(RAW, parse_dates=['date'])
    df.columns = df.columns.str.strip()
    for col in df.columns:
        if col == 'date':
            continue
        if df[col].dtype == object:
            df[col] = pd.to_numeric(df[col].astype(str).str.strip(), errors='coerce')
    return df.sort_values('date').reset_index(drop=True)

df = load_raw()
print('Shape:', df.shape)
print('Date range:', df['date'].min(), '→', df['date'].max())
print('Missing:', df.isnull().sum().sum())
print('Duplicate dates:', df['date'].duplicated().sum())
df.head()

## Step 01 — Distribution audit

In [ ]:
num_rows = []
for col in df.columns:
    if col == 'date':
        continue
    s = df[col].astype(float)
    num_rows.append({
        'column': col,
        'missing': int(s.isna().sum()),
        'skew': round(float(stats.skew(s)), 3),
        'zero_pct': round(100 * (s == 0).mean(), 1),
        'min': round(float(s.min()), 2),
        'max': round(float(s.max()), 2),
    })
num_prof = pd.DataFrame(num_rows)

diffs = df['date'].diff().dropna()
mode_min = int(diffs.mode().iloc[0].total_seconds() / 60)
quality_pass = (
    df.isnull().sum().sum() == 0
    and df['date'].duplicated().sum() == 0
    and (diffs != diffs.mode().iloc[0]).sum() == 0
)

print('Quality gate:', 'PASS' if quality_pass else 'FAIL')
print('Interval (min):', mode_min)
print('Target skew:', round(stats.skew(df[TARGET]), 3))
if 'rv1' in df.columns:
    print('rv1==rv2 all rows:', (df['rv1'] == df['rv2']).all())
display(num_prof.head(12))

# plots
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df[TARGET], bins=60, edgecolor='white')
ax.set_title(f'{TARGET} distribution')
plt.show()

df2 = df.copy()
df2['hour'] = df2['date'].dt.hour
df2.groupby('hour')[TARGET].mean().plot(marker='o', figsize=(10, 3), title='Mean Appliances by hour')
plt.show()

step01_report = {
    'quality_pass': quality_pass,
    'n_rows': len(df),
    'interval_min': mode_min,
    'target_skew': round(float(stats.skew(df[TARGET])), 4),
}
(OUT / 'step01' / 'step01_report.json').write_text(json.dumps(step01_report, indent=2))
num_prof.to_csv(OUT / 'step01' / 'numeric_profile.csv', index=False)
print('Saved step01 outputs')

## Step 02 — Correlation & drop list

In [ ]:
from numpy.linalg import LinAlgError

input_cols = [c for c in df.columns if c not in ('date', TARGET)]
corr_ff = df[input_cols].corr()

pairs = []
for i, a in enumerate(input_cols):
    for b in input_cols[i + 1:]:
        r = df[a].corr(df[b])
        if abs(r) >= HIGH_R_THRESHOLD:
            pairs.append({'feature_a': a, 'feature_b': b, 'r': round(r, 4)})
pairs_df = pd.DataFrame(pairs).sort_values('r', key=abs, ascending=False)

corr_target = df[[TARGET] + input_cols].corr()[TARGET].drop(TARGET)
corr_target = corr_target.loc[corr_target.abs().sort_values(ascending=False).index]

# decide drops
drop = set(MUST_DROP)
drop_reason = {
    'rv1': 'Random UCI placeholder',
    'rv2': 'Random UCI placeholder',
}
for _, row in pairs_df.iterrows():
    a, b, r = row['feature_a'], row['feature_b'], row['r']
    if {a, b} == {'T6', 'T_out'}:
        drop.add('T6')
        drop_reason['T6'] = f'Near-duplicate of T_out (r={r})'
    if 'T9' in (a, b) and abs(r) > 0.9:
        drop.add('T9')
        drop_reason['T9'] = f'Correlated room sensor (r={r})'

drop_list = sorted(drop)
leakage = {k: round(float(v), 4) for k, v in corr_target.items() if abs(v) >= LEAKAGE_R_THRESHOLD}

print('Leakage features (|r|>=0.95):', leakage or 'none')
print('DROP:', drop_list)
print('\nTop correlations with target:')
display(corr_target.head(8))

plt.figure(figsize=(10, 8))
sns.heatmap(corr_ff, cmap='RdBu_r', center=0)
plt.title('Feature-feature correlation')
plt.tight_layout()
plt.show()

step02_decisions = {
    'proposed_drop': drop_list,
    'drop_reason': drop_reason,
    'leakage_candidates': leakage,
}
(OUT / 'step02' / 'step02_report.json').write_text(
    json.dumps({'decisions': step02_decisions, 'target_correlation': corr_target.to_dict()}, indent=2))
pairs_df.to_csv(OUT / 'step02' / 'high_corr_pairs.csv', index=False)
print('Saved step02 outputs')

## Step 03 — Preprocess (10min + hourly)

In [ ]:
def add_time_features(d):
    out = d.copy()
    hour = out['date'].dt.hour + out['date'].dt.minute / 60.0
    dow = out['date'].dt.dayofweek
    out['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    out['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    out['dow_sin'] = np.sin(2 * np.pi * dow / 7)
    out['dow_cos'] = np.cos(2 * np.pi * dow / 7)
    return out

def to_hourly(d):
    x = d.set_index('date')
    agg = {c: ('sum' if c in SUM_COLS else 'mean') for c in x.columns}
    return x.resample('h').agg(agg).reset_index()

def save_track(d, track, drops):
    work = d.drop(columns=[c for c in drops if c in d.columns])
    ordered = [TARGET] + [c for c in work.columns if c not in ('date', TARGET)]
    work = work[['date'] + ordered]
    n = len(work)
    split_idx = int(n * (1 - TEST_RATIO))
    train_df, test_df = work.iloc[:split_idx], work.iloc[split_idx:]
    scaler = MinMaxScaler()
    scaler.fit(train_df[ordered])
    train_s, test_s = train_df.copy(), test_df.copy()
    train_s[ordered] = scaler.transform(train_df[ordered])
    test_s[ordered] = scaler.transform(test_df[ordered])
    full = pd.concat([train_s, test_s], ignore_index=True)
    prep = OUT / 'preprocess' / track
    prep.mkdir(parents=True, exist_ok=True)
    full.drop(columns=['date']).to_csv(prep / 'data.csv', index=False)
    with open(prep / 'scaler.pkl', 'wb') as f:
        pickle.dump(scaler, f)
    meta = {
        'track': track, 'target': TARGET, 'feature_names': ordered,
        'n_features': len(ordered), 'n_rows': n, 'split_idx': split_idx,
        'dropped_features': drops,
    }
    (prep / 'meta.json').write_text(json.dumps(meta, indent=2))
    return meta

df = add_time_features(load_raw())
work = df.drop(columns=[c for c in drop_list if c in df.columns])
ordered_pre = [TARGET] + [c for c in work.columns if c not in ('date', TARGET)]
post_corr = work[ordered_pre].corr()[TARGET].drop(TARGET)
validation_pass = all(abs(v) < 0.95 for v in post_corr)

meta_10 = save_track(df, '10min', drop_list)
meta_h = save_track(to_hourly(df), 'hourly', drop_list)

print('Validation pass:', validation_pass)
print('10min:', meta_10['n_rows'], 'rows,', meta_10['n_features'], 'features')
print('hourly:', meta_h['n_rows'], 'rows,', meta_h['n_features'], 'features')
print('Features:', meta_10['feature_names'])

plt.figure(figsize=(10, 8))
sns.heatmap(work[ordered_pre].corr(), annot=False, cmap='RdBu_r', center=0)
plt.title('Correlation after drops (unscaled)')
plt.tight_layout()
plt.show()

manifest = {'dropped': drop_reason, 'kept': meta_10['feature_names'], 'tracks': {'10min': meta_10, 'hourly': meta_h}}
(OUT / 'step03' / 'feature_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print('Saved preprocess/10min and preprocess/hourly')

## Step 04 — Window validation

In [ ]:
def build_windows(arr, window, target_idx=0):
    X, y = [], []
    for i in range(len(arr) - window):
        X.append(arr[i:i + window])
        y.append(arr[i + window, target_idx])
    return np.array(X, np.float32), np.array(y, np.float32)

report = {'windows': WINDOWS, 'tracks': {}}
for track, wins in WINDOWS.items():
    meta = json.loads((OUT / 'preprocess' / track / 'meta.json').read_text())
    cols = meta['feature_names']
    arr = pd.read_csv(OUT / 'preprocess' / track / 'data.csv')[cols].to_numpy(np.float32)
    tidx = cols.index(TARGET)
    report['tracks'][track] = {}
    for w in wins:
        X, _ = build_windows(arr, w, tidx)
        split = int(len(X) * (1 - TEST_RATIO))
        ok = split > 100 and (len(X) - split) > 50
        report['tracks'][track][str(w)] = {'n_samples': len(X), 'n_train': split, 'ok': ok}
        print(f'{track} win{w:>3}: {len(X)} samples [{"OK" if ok else "LOW"}]')

(OUT / 'step04' / 'windows_report.json').write_text(json.dumps(report, indent=2))
print('\nDone. Next: train_10min.ipynb and train_hourly.ipynb')

## Done

Saved under `outputs/`:
- `preprocess/10min/data.csv` + `scaler.pkl`
- `preprocess/hourly/data.csv` + `scaler.pkl`
- `step01/` … `step04/` audit files

Run training notebooks next.